<div dir="rtl">
<h2>Streaming: קליטה ועיבוד של נתונים מתמשכים</h2>

<p>מחברת הקורס של ד"ר פאול אברהמי. נעקוב אחר אירועי חנות דמיונית מרגע יצירתם עד לטבלאות הכנסות ולהתראות. כל הנתונים סינתטיים.</p>
<p><b>הרצה ראשונה:</b> מתוך תיקיית הפרויקט מריצים בטרמינל <code dir="ltr">uv sync --python 3.13.12</code>. ב-Cursor פותחים מחברת זו ובוחרים את סביבת הפרויקט תחת Select Kernel. לאחר מכן לוחצים Run All.</p>
<p>המסלול הראשי פועל מקומית ללא Docker, ללא חשבון ענן וללא מפתח API. היצרן והצרכן רצים בו בזמן דרך תור מוגבל בזיכרון. בסוף מופיע מסלול רשות עם ברוקר אמיתי התואם ל-Kafka.</p>
<p>בכל שלב קוראים את ההסבר, מנבאים תוצאה, מריצים את התא ובודקים את הטבלה שמופיעה מתחתיו. הפעלת כל התאים כוללת הרצה חיה קצרה ויוצרת תיקיית תוצאות חדשה.</p>

</div>

<div dir="rtl">
<h2>שלב 0: סביבת ההרצה ותיקיית התוצאות</h2>

<p><b>מטרה:</b> לוודא שהמחברת מריצה את Python של הפרויקט. Kernel הוא תהליך Python שמריץ את התאים ושומר את המשתנים ביניהם.</p>
<p><b>תוצאה:</b> גרסת Python, מיקום המפרש ותיקיית עבודה ייחודית. בהרצה חוזרת נוצרת תיקייה חדשה תחת <code dir="ltr">lab_runs</code>. אין מחיקה של תוצאות קודמות.</p>

</div>

In [ ]:
from pathlib import Path
from datetime import timedelta
from itertools import count
import asyncio
import json
import sqlite3
import sys
from IPython.display import HTML, FileLink, display
from streaming_lab import (
    Delivery, Event, StreamEngine, InjectedCrash, teaching_fixture,
    generate_order_fixture, generate_review_fixture, batch_truth, run_live, run_microbatch,
)
from streaming_lab.events import T0
from streaming_lab.reporting import fresh_run_directory, money, canonical_results
from streaming_lab.notebook_tools import show_table, table_html, save_report

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "pyproject.toml").exists() and (p / "src/streaming_lab").is_dir()), None)
if ROOT is None:
    raise RuntimeError("פתחו את המחברת מתוך תיקיית streaming-course-demo")
assert sys.version_info[:3] == (3, 13, 12), "בחרו Kernel של Python 3.13.12"
RUN_DIR = fresh_run_directory(ROOT / "lab_runs", prefix="notebook-").resolve()
checks = []
def check(name, condition):
    assert condition, name
    checks.append({"בדיקה": name, "תוצאה": "עבר"})
    print("PASS:", name)
def new_db(name):
    return fresh_run_directory(RUN_DIR, prefix=name + "-") / "stream.sqlite"

show_table([
    {"פרט": "Python", "ערך": sys.version.split()[0]},
    {"פרט": "מפרש המחברת", "ערך": sys.executable},
    {"פרט": "תיקיית התוצאות", "ערך": str(RUN_DIR)},
])
check("גרסת Python היא 3.13.12", sys.version_info[:3] == (3, 13, 12))


<div dir="rtl">
<h2>שלב 1: הודעה, אירוע ומיקום בלוג</h2>

<p><b>קלט:</b> 17 מסירות קטנות שאפשר לבדוק ידנית. לכל מסירה יש offset, ולכל אירוע יש event_id וזמן אירוע.</p>
<p><b>ההבדל:</b> אותו אירוע יכול להגיע שוב במיקום חדש. offset מציין מיקום במסלול הקלט, ואילו event_id משמש לזיהוי האירוע העסקי.</p>
<p>הסכום נשמר באגורות כמספר שלם. כך 1,990 אגורות הן 19.90 שקלים, בלי שגיאת עיגול מצטברת של מספרים עשרוניים בינאריים.</p>

</div>

In [ ]:
small = teaching_fixture()
input_rows = []
for item in small:
    payload = item.event.to_dict() if isinstance(item.event, Event) else item.event
    input_rows.append({"offset": item.offset, **payload})
show_table(input_rows, [
    ("offset","מיקום"), ("event_id","מזהה אירוע"), ("kind","סוג"),
    ("event_time","זמן האירוע"), ("amount_cents","אגורות"), ("rating","דירוג"),
], caption="הקלט לפי סדר ההגעה")
check("הדוגמה מכילה 17 מסירות", len(small) == 17)
print(json.dumps(input_rows[0], ensure_ascii=False, indent=2))


<div dir="rtl">
<h2>שלב 2: חישוב מלא לצורך השוואה</h2>

<p><b>פעולה:</b> מחשבים את כל האירועים התקינים והייחודיים בלי להתחשב בסדר ההגעה. זהו חישוב batch של אותו קלט.</p>
<p><b>צפוי:</b> 9 הזמנות בסך 135.00 שקלים, 5 ביקורות תקינות ורשומה אחת שאינה עומדת בסכמה. החישוב אינו מוסיף פעמיים את אותה זהות אירוע.</p>
<p>זהו נתון להשוואה של ההדגמה. במערכת אמיתית גם חישוב batch דורש מקורות שלמים וכללי איכות נכונים.</p>

</div>

In [ ]:
small_truth = batch_truth(small)
show_table([{
    "הזמנות ייחודיות": small_truth["orders"],
    "הכנסה": money(small_truth["revenue_cents"]),
    "ביקורות": small_truth["reviews"],
    "רשומות לא תקינות": small_truth["invalid"],
}])
check("חישוב הייחוס הוא 9 הזמנות ו-135.00 שקלים",
      (small_truth["orders"], small_truth["revenue_cents"]) == (9, 13500))


<div dir="rtl">
<h2>שלב 3: עיבוד אירוע אחד בכל פעם</h2>

<p><b>פעולה:</b> מעבירים למנוע כל הודעה בנפרד. אחרי כל הודעה מסתכלים על ההחלטה, ה-watermark והחלונות הפתוחים והסגורים.</p>
<p>חלון דקה כולל את תחילתו ואינו כולל את סופו. אירוע בשעה 10:01:00 שייך לחלון שמתחיל ב-10:01:00.</p>
<p>בהרצה זו ההשהיה של ה-watermark היא אפס, וה-grace הוא 10 שניות. חלון נסגר כאשר ה-watermark מגיע לסוף החלון ועוד 10 שניות. אירוע עבור חלון שכבר נסגר נשמר במסלול המאוחרים.</p>

</div>

In [ ]:
teaching_db = new_db("teaching")
trace = []
with StreamEngine(teaching_db, grace_seconds=10) as engine:
    for item in small:
        outcome = engine.ingest(item.event, offset=item.offset)
        snap = engine.snapshot()
        trace.append({
            "offset": item.offset, "החלטה": outcome["status"],
            "watermark": snap["watermark"],
            "חלונות הכנסה פתוחים": len(snap["open_windows"]),
            "חלונות הכנסה שנסגרו": len(snap["revenue_windows"]),
            "checkpoint": snap["checkpoint"],
        })
    teaching_before_end = engine.snapshot()
    teaching = engine.finalize_demo()
show_table(trace, caption="המצב אחרי כל הודעה")


<div dir="rtl">
<h2>שלב 4: קריאת התוצרים והבדיקות</h2>

<p><b>צפוי:</b> 7 הזמנות מתקבלות, בסך 122.00 שקלים. שתי הזמנות מאוחרות, בסך 13.00 שקלים, נשמרות בנפרד. שתי מסירות כפולות אינן מכפילות את הסכום. הרשומה עם סכום מסוג float נשמרת בטבלת השגיאות.</p>
<p><b>סיום הדגמה:</b> הקריאה ל-finalize_demo מצהירה שכל הקלט הסופי הגיע וסוגרת את החלונות שנותרו. היא אינה מזיזה את ה-watermark. שתיקה בזרם חי אינה הצהרת סיום.</p>

</div>

In [ ]:
show_table(teaching["revenue_windows"], [
    ("window_start","תחילת החלון"), ("window_end","סוף החלון"),
    ("orders","הזמנות"), ("revenue_cents","הכנסה באגורות"), ("closure_reason","סיבת הסגירה"),
], caption="ארבעת חלונות ההכנסות")
show_table(teaching["late_events"], [
    ("event_id","מזהה"), ("source_offset","מיקום בקלט"),
    ("event_time","זמן האירוע"), ("amount_cents","אגורות"),
], caption="אירועים מאוחרים שנשמרו")
show_table(teaching["dead_letters"], [
    ("source_offset","מיקום בקלט"), ("reason","סיבת השגיאה"), ("payload","הקלט המקורי"),
], caption="רשומות לא תקינות")
check("ההכנסה שהתקבלה היא 122.00 שקלים", teaching["totals"]["revenue_cents"] == 12200)
check("13.00 שקלים נשמרו במסלול המאוחרים", teaching["totals"]["late_order_cents"] == 1300)
check("122.00 ועוד 13.00 שווים לחישוב המלא",
      teaching["totals"]["revenue_cents"] + teaching["totals"]["late_order_cents"] == small_truth["revenue_cents"])
check("זוהו שתי כפילויות ורשומה אחת לא תקינה",
      (teaching["stats"]["duplicates"], teaching["stats"]["invalid"]) == (2, 1))
check("סיום הקלט אינו מקדם את ה-watermark",
      teaching["watermark_ms"] == teaching_before_end["watermark_ms"])
check("נשמרה נקודת ההמשך לאחר כל 17 המסירות", teaching["checkpoint"] == 16)


<div dir="rtl">
<h2>שלב 5: שני פרמטרים נפרדים לזמן</h2>

<p>במנוע שלנו ה-watermark נגזר מזמן האירוע המרבי מבין אירועים תקינים וייחודיים שהתקבלו לעיבוד, פחות out_of_order_seconds. הוא לעולם אינו נסוג. כפילויות ורשומות שסווגו כלא תקינות או מאוחרות אינן מקדמות אותו. grace_seconds מוסיף זמן קבלה נוסף אחרי סוף החלון.</p>
<p>בדוגמה הבאה זמן האירוע המרבי הוא 10:01:29. השהיה של 20 שניות מציבה את ה-watermark ב-10:01:09. עם grace של 10 שניות, החלון הראשון עדיין פתוח. אירוע ב-10:01:30 מקדם את ה-watermark ל-10:01:10 וסוגר אותו.</p>

</div>

In [ ]:
def simple_order(identity, seconds, cents=100):
    return Event(identity, "order", T0 + timedelta(seconds=seconds), 5, amount_cents=cents)
with StreamEngine(new_db("watermark"), out_of_order_seconds=20, grace_seconds=10) as engine:
    engine.ingest(simple_order("newer", 89), offset=0)
    a = engine.snapshot()
    accepted_older = engine.ingest(simple_order("older", 5), offset=1)
    engine.ingest(simple_order("newest", 90), offset=2)
    b = engine.snapshot()
    rejected_older = engine.ingest(simple_order("too-late", 6), offset=3)
show_table([
    {"מצב": "לפני קידום השעון", "watermark": a["watermark"], "החלטה על האירוע הישן": accepted_older["status"]},
    {"מצב": "אחרי קידום השעון", "watermark": b["watermark"], "החלטה על האירוע הישן": rejected_older["status"]},
])
check("אירוע ישן מתקבל לפני הסגירה ונשמר כמאוחר אחריה",
      accepted_older["status"] == "accepted" and rejected_older["status"] == "late")


<div dir="rtl">
<h2>שלב 6: השוואת מדיניות האיחור על 300 הזמנות</h2>

<p><b>קלט:</b> מחולל קבוע יוצר 300 הזמנות ייחודיות ו-17 מסירות כפולות. סדר ההגעה כולל איחורים. זמן האירוע הוא זמן מדומה קבוע, ולכן אפשר להשוות בין מחשבים.</p>
<p><b>ניסוי:</b> מריצים אותו קלט עם grace של 0, 10 ו-120 שניות. השהיית ה-watermark נשארת אפס. לכל ניסוי נוצר מסד נתונים נפרד.</p>
<p><b>ניבוי:</b> הגדלת ה-grace מאפשרת לקבל יותר מהאירועים המאוחרים של הקלט הזה, אך דוחה את סגירת החלונות. היא אינה מבטיחה שלא יהיו איחורים בכל מקור עתידי.</p>

</div>

In [ ]:
orders_input = generate_order_fixture(n=300, seed=7)
orders_truth = batch_truth(orders_input)
scenarios, comparison = {}, []
for grace in (0, 10, 120):
    db_path = new_db(f"grace-{grace}")
    with StreamEngine(db_path, grace_seconds=grace) as engine:
        for item in orders_input:
            engine.ingest(item.event, offset=item.offset)
        result = engine.finalize_demo()
    scenarios[grace] = {"path": db_path, "snapshot": result}
    comparison.append({
        "grace_seconds": grace,
        "accepted_orders": result["totals"]["orders"],
        "accepted_revenue": money(result["totals"]["revenue_cents"]),
        "accepted_cents": result["totals"]["revenue_cents"],
        "late_orders": result["totals"]["late_orders"],
        "late_cents": result["totals"]["late_order_cents"],
        "duplicate_deliveries": result["stats"]["duplicates"],
    })
show_table(comparison, [
    ("grace_seconds","Grace בשניות"), ("accepted_orders","הזמנות שהתקבלו"),
    ("accepted_revenue","הכנסה שהתקבלה"), ("late_orders","הזמנות מאוחרות"),
    ("duplicate_deliveries","מסירות כפולות"),
])
check("קלט ההזמנות מכיל 317 מסירות", len(orders_input) == 317)
check("חישוב הייחוס הוא 61,241.70 שקלים", orders_truth["revenue_cents"] == 6124170)
check("תוצאות החלונות תואמות לקלט הקבוע",
      [r["accepted_orders"] for r in comparison] == [279, 286, 300])


<div dir="rtl">
<h2>שלב 7: בדיקת הסכומים ישירות במסד הנתונים</h2>

<p>בדיקת הודעת הצלחה אינה מספיקה. נפתח חיבור חדש ל-SQLite ונקרא את הטבלה שנשמרה בדיסק.</p>
<p>שאילתת הסכום פועלת לאחר הסגירה המפורשת של כל חלונות הדוגמה הסופית. בזמן זרם חי חלק מהנתונים נמצאים עדיין בטבלת המצב של החלונות הפתוחים.</p>

</div>

In [ ]:
orders = scenarios[10]["snapshot"]
with sqlite3.connect(scenarios[10]["path"]) as con:
    con.row_factory = sqlite3.Row
    sql_totals = dict(con.execute(
        "SELECT SUM(orders) AS orders, SUM(revenue_cents) AS revenue_cents FROM realtime_revenue"
    ).fetchone())
show_table(orders["revenue_windows"], [
    ("window_start","תחילת החלון"), ("orders","מספר הזמנות"),
    ("revenue_cents","הכנסה באגורות"), ("closure_reason","סיבת הסגירה"),
])
show_table([sql_totals], caption="סכום שחושב מחדש מתוך הטבלה בדיסק")
check("הטבלה בדיסק מכילה 286 הזמנות", sql_totals["orders"] == 286)
check("הסכום שנקרא מהדיסק תואם לתוצאת המנוע",
      sql_totals["revenue_cents"] == orders["totals"]["revenue_cents"])


<div dir="rtl">
<h2>שלב 8: אירועים מאוחרים והתאמה לחישוב המלא</h2>

<p>בגרסת grace של 10 שניות נשמרות 14 הזמנות מאוחרות. הן אינן משנות חלון סגור, אבל הנתונים המקוריים שלהן נשארים זמינים לבירור ולחישוב מתקן.</p>
<p>נבדוק שההזמנות שהתקבלו וההזמנות המאוחרות משלימות בדיוק את חישוב ה-batch, הן במספר ההזמנות והן בסכום. זו בדיקת התאמה חשובה יותר ממספר השורות לבדו.</p>

</div>

In [ ]:
show_table(orders["late_events"], [
    ("event_id","מזהה"), ("event_time","זמן האירוע"),
    ("watermark","Watermark בעת הקליטה"), ("amount_cents","אגורות"),
], limit=6, caption="דגימה של האירועים המאוחרים")
for grace, scenario in scenarios.items():
    totals = scenario["snapshot"]["totals"]
    check(f"סכום ההזמנות נסגר בדיוק עם grace={grace}",
          totals["revenue_cents"] + totals["late_order_cents"] == orders_truth["revenue_cents"])
    check(f"מספר ההזמנות נסגר בדיוק עם grace={grace}",
          totals["orders"] + totals["late_orders"] == orders_truth["orders"])


<div dir="rtl">
<h2>שלב 9: קריאה חוזרת של כל הקלט</h2>

<p>אחרי עצירה, צרכן עשוי לקבל שוב הודעות שכבר עיבד. המנוע שומר את נקודת ההמשך ואת השפעת האירוע באותה עסקה מקומית.</p>
<p>נפתח את המסד מחדש ונעביר שוב את כל 317 המסירות. המיקומים שכבר נשמרו יזוהו כ-replayed. התוצאות והספירות העסקיות צריכות להישאר זהות.</p>

</div>

In [ ]:
before_replay = canonical_results(orders)
with StreamEngine(scenarios[10]["path"], grace_seconds=10) as engine:
    statuses = [engine.ingest(item.event, offset=item.offset)["status"] for item in orders_input]
    after_replay = engine.snapshot()
check("כל 317 המיקומים זוהו כקריאה חוזרת", statuses == ["replayed"] * 317)
check("קריאה חוזרת אינה משנה את התוצאה", canonical_results(after_replay) == before_replay)
show_table([{
    "מסירות שנקראו שוב": len(statuses),
    "הזמנות לפני": orders["totals"]["orders"],
    "הזמנות אחרי": after_replay["totals"]["orders"],
    "הכנסה אחרי": money(after_replay["totals"]["revenue_cents"]),
}])


<div dir="rtl">
<h2>שלב 10: הזרקת קריסה לפני שמירה ואחריה</h2>

<p><b>לפני commit:</b> העסקה מתבטלת, ואותו אירוע צריך להתעבד שוב לאחר פתיחת המסד מחדש.</p>
<p><b>אחרי commit:</b> השפעת האירוע כבר נשמרה, ולכן הקריאה החוזרת אינה משנה את הסכום. נבדוק את שני המצבים באותו offset.</p>
<p>המחברת מזריקה חריגה וסוגרת את התהליך הלוגי. בדיקות הפרויקט כוללות גם תהליך נפרד שמסתיים בלי cleanup. ההבטחה כאן מוגבלת לעסקה בתוך SQLite. commit של offset ב-Kafka הוא פעולה נפרדת.</p>

</div>

In [ ]:
recovery_rows = []
for phase in ("before_commit", "after_commit"):
    path = new_db("crash-" + phase)
    crash_offset = 8
    with StreamEngine(path) as engine:
        try:
            for item in small:
                engine.ingest(item.event, offset=item.offset,
                              crash_at=phase if item.offset == crash_offset else None)
        except InjectedCrash:
            checkpoint_at_crash = engine.snapshot()["checkpoint"]
        else:
            raise AssertionError("הקריסה המתוכננת לא התרחשה")
    with StreamEngine(path) as restarted:
        retry_status = restarted.ingest(small[crash_offset].event, offset=crash_offset)["status"]
        for item in small[crash_offset + 1:]:
            restarted.ingest(item.event, offset=item.offset)
        recovered = restarted.finalize_demo()
    check(f"התאוששות מ-{phase} זהה להרצה מלאה",
          canonical_results(recovered) == canonical_results(teaching))
    recovery_rows.append({
        "מיקום הקריסה": phase, "checkpoint בעת הקריסה": checkpoint_at_crash,
        "תוצאת הניסיון החוזר": retry_status,
        "הכנסה בסיום": money(recovered["totals"]["revenue_cents"]),
    })
show_table(recovery_rows)


<div dir="rtl">
<h2>שלב 11: הזמנות מגיעות ומעובדות ברצף</h2>

<p><b>כאן מופעל זרם הזמנות חי מקומי:</b> מחולל יוצר הזמנה חדשה בכל פעם שהיצרן מבקש אותה. אין קובץ יומי ואין רשימה של כל ההזמנות שנטענה מראש. היצרן ממשיך למסור הזמנות לאורך זמן, והצרכן קורא אותן מתור מוגבל ומעבד כל אחת כשהיא זמינה.</p>
<p>נעצור את יצירת האירועים אחרי שלוש שניות ונרוקן את ההודעות שכבר נכנסו לתור. מספר ההודעות בפועל תלוי במהירות המחשב, ולכן אינו מספר צפוי קבוע.</p>
<p>שעון האירועים הסינתטי מואץ בשתי שניות לכל מסירה כדי להראות חלונות בזמן קצר. זמן העיבוד הוא זמן שעון המערכת בעת הטיפול באירוע, ואילו elapsed_s מודד את משך הניסוי. המחברת משתמשת ב-await כי Jupyter כבר מפעיל event loop.</p>

</div>

In [ ]:
def continuous_orders():
    for i in count():
        yield Delivery(i, Event(
            f"continuous-order-{i}", "order", T0 + timedelta(seconds=2 * i),
            5, amount_cents=1000 + 10 * (i % 10),
        ))

live_display = display(HTML('<p dir="rtl">היצרן והצרכן מתחילים לפעול</p>'), display_id=True)
def show_live(snapshot):
    if snapshot["stats"]["received"] % 10 == 0:
        queue = snapshot.get("queue", {})
        live_display.update(HTML(table_html([{
            "עובדו": snapshot["stats"]["received"],
            "בתור עכשיו": queue.get("depth", ""),
            "קיבולת התור": queue.get("maxsize", ""),
            "הזמנות": snapshot["totals"]["orders"],
            "הכנסה": money(snapshot["totals"]["revenue_cents"]),
            "watermark": snapshot["watermark"],
        }])))
with StreamEngine(new_db("live")) as engine:
    live = await run_live(engine, continuous_orders(), duration_s=3.0, interval_s=0.03,
                          processing_delay_s=0.015, queue_maxsize=8,
                          on_snapshot=show_live)
    live_snapshot = engine.snapshot()
show_table([{k:live[k] for k in (
    "produced", "consumed", "elapsed_s", "max_queue_depth", "backpressure_waits", "stop_reason"
)}])
check("כל ההודעות שהתקבלו לתור עובדו לפני העצירה", live["produced"] == live["consumed"] > 0)
check("התור נשאר בגבול הקיבולת שלו", live["max_queue_depth"] <= 8)
check("עצירת השעון אינה מסיימת את חלונות הזרם", live_snapshot["finalized"] is False)
check("המקור החי בדוגמה זו מייצר רק הזמנות", live_snapshot["totals"]["reviews"] == 0)


<div dir="rtl">
<h2>שלב 11ב: רשומה בודדת לעומת קבוצות קטנות</h2>

<p><b>אותו זרם, שתי שיטות עיבוד:</b> נעביר 12 הזמנות באותו סדר ובאותו קצב בשתי ריצות נפרדות. בריצה הראשונה כל הזמנה מעובדת כשהיא זמינה. בריצה השנייה הצרכן אוסף עד חמש הזמנות או ממתין עד חצי שנייה מאז תחילת האיסוף, לפי המוקדם.</p>
<p>גם כאן היצרן מוסר הזמנה אחת בכל פעם לתור אסינכרוני. רשימת הקלט הסופית משמשת להשוואה ניתנת לשחזור; הצרכן אינו מקבל אותה בשלמותה מראש. תצוגת הקבוצות מתעדכנת בסיום כל קבוצה. בדרך כלל נראה קבוצות של 5, 5 ו-2; בטבלה מוצגים הגדלים שנצפו בפועל.</p>
<p><b>הגבולות:</b> כל רשומה נשמרת בעסקה נפרדת, גם בתוך קבוצה. אין כאן התחייבות לכתיבה אטומית של כל הקבוצה או לשיפור מהירות. חלון דקה לפי זמן אירוע הוא כלל צבירה נפרד מקבוצת העיבוד.</p>

</div>

In [ ]:
paced_orders = [
    Delivery(i, Event(f"compare-{i}", "order", T0 + timedelta(seconds=2 * i),
                      5, amount_cents=100 + i))
    for i in range(12)
]
individual_offsets = []
microbatch_trace = []
individual_display = display(HTML('<p dir="rtl">עיבוד רשומות בודדות מתחיל</p>'), display_id=True)
group_display = display(HTML('<p dir="rtl">איסוף קבוצות מתחיל</p>'), display_id=True)
def show_individual(snapshot):
    individual_offsets.append(snapshot["checkpoint"])
    individual_display.update(HTML(table_html([{
        "שיטה": "רשומה בודדת", "עובדו": snapshot["totals"]["orders"],
        "הכנסה": money(snapshot["totals"]["revenue_cents"]),
    }])))
def show_group(snapshot):
    batch = snapshot["batch"]
    microbatch_trace.append({
        "קבוצה": batch["index"], "גודל בפועל": batch["size"],
        "סיבת הפעלה": batch["flush_reason"],
        "זמן איסוף בשניות": round(batch["collection_wait_s"], 3),
        "offset ראשון": batch["first_offset"], "offset אחרון": batch["last_offset"],
        "הזמנות שעובדו": snapshot["totals"]["orders"],
        "הכנסה": money(snapshot["totals"]["revenue_cents"]),
    })
    group_display.update(HTML(table_html(microbatch_trace)))

with StreamEngine(new_db("individual-orders")) as engine:
    individual = await run_live(engine, paced_orders, duration_s=None,
                                interval_s=0.04, on_snapshot=show_individual)
with StreamEngine(new_db("microbatch-orders")) as engine:
    grouped = await run_microbatch(engine, paced_orders, duration_s=None,
                                  interval_s=0.04, max_batch_size=5, max_wait_s=0.5,
                                  on_snapshot=show_group)
microbatch_comparison = [
    {"שיטה": "רשומה בודדת", "הזמנות": individual["snapshot"]["totals"]["orders"],
     "עדכוני תצוגה": len(individual_offsets), "גדלי הקבוצות": "1 בכל עדכון",
     "הכנסה": money(individual["snapshot"]["totals"]["revenue_cents"])},
    {"שיטה": "קבוצות קטנות", "הזמנות": grouped["snapshot"]["totals"]["orders"],
     "עדכוני תצוגה": grouped["batches"], "גדלי הקבוצות": grouped["batch_sizes"],
     "הכנסה": money(grouped["snapshot"]["totals"]["revenue_cents"])},
]
show_table(microbatch_comparison)
show_table(microbatch_trace, caption="נקודות העדכון שנצפו בריצה")
check("אותן הזמנות מניבות אותו מצב עסקי בשתי השיטות",
      canonical_results(individual["snapshot"]) == canonical_results(grouped["snapshot"]))
check("בשתי השיטות התקבלו 12 הזמנות בסך 12.66 שקלים",
      grouped["snapshot"]["totals"]["orders"] == 12
      and grouped["snapshot"]["totals"]["revenue_cents"] == 1266)
check("כל רשומה עובדה וכל קבוצה נשארה בגבול הגודל",
      sum(grouped["batch_sizes"]) == 12 and max(grouped["batch_sizes"]) <= 5)
check("סיום איסוף הקבוצות אינו סיום חלונות זמן האירוע",
      grouped["snapshot"]["finalized"] is False)


<div dir="rtl">
<h2>שלב 11ג: קבוצה יוצאת גם כשהיא אינה מלאה</h2>

<p>כעת המקור איטי: הזמנה מגיעה בכל 0.12 שנייה, אבל זמן האיסוף המרבי הוא 0.04 שנייה. הצרכן צריך לעבד גם קבוצה קטנה, בלי לחכות לעוד ארבע הזמנות.</p>
<p>בטבלה נבדוק את סיבת ההפעלה. max_size פירושו שהקבוצה התמלאה, max_wait פירושו שזמן האיסוף חלף, ו-final_drain פירושו שהמקור הודיע שסיים ונשארו רשומות. זמן האיסוף אינו כולל את כל זמן ההמתנה בתור ואת זמן העיבוד, ולכן אינו הבטחה לזמן התגובה הכולל.</p>

</div>

In [ ]:
with StreamEngine(new_db("microbatch-timer")) as engine:
    timed_group = await run_microbatch(
        engine, paced_orders[:3], duration_s=None, interval_s=0.12,
        max_batch_size=5, max_wait_s=0.04,
    )
show_table(timed_group["batch_details"], [
    ("index","קבוצה"), ("size","גודל בפועל"), ("flush_reason","סיבת הפעלה"),
    ("collection_wait_s","זמן איסוף בשניות"),
])
check("גם מקור איטי מעובד בלי להמתין לקבוצה מלאה",
      timed_group["consumed"] == 3 and timed_group["flush_counts"]["max_wait"] > 0)
check("אין קבוצה ריקה או קבוצה גדולה מהגבול",
      all(1 <= size <= 5 for size in timed_group["batch_sizes"]))


<div dir="rtl">
<h2>שלב 12: צרכן איטי ולחץ חוזר</h2>

<p><b>ניסוי:</b> היצרן הבא מכניס 60 מסירות במהירות, והצרכן מחכה מעט לפני כל עיבוד. קיבולת התור היא שלוש הודעות בלבד.</p>
<p>כשהתור מלא, await queue.put ממתין. זהו לחץ חוזר: האטת המקור במקום גדילה בלתי מוגבלת בזיכרון. אחרי עצירת המקור מסיימים לעבד את כל מה שכבר נכנס.</p>

</div>

In [ ]:
with StreamEngine(new_db("slow-consumer")) as engine:
    pressure = await run_live(
        engine, orders_input[:60], duration_s=None,
        interval_s=0, processing_delay_s=0.015, queue_maxsize=3,
    )
show_table([{k:pressure[k] for k in (
    "produced", "consumed", "queue_maxsize", "max_queue_depth",
    "backpressure_waits", "producer_wait_s", "elapsed_s"
)}])
check("הצרכן האיטי עיבד את כל 60 המסירות", pressure["consumed"] == 60)
check("עומק התור אינו גדול משלוש", pressure["max_queue_depth"] <= 3)
check("היצרן נאלץ להמתין בגלל התור המלא", pressure["backpressure_waits"] > 0)


<div dir="rtl">
<h2>שלב 13: התראה על ביקורות שליליות</h2>

<p><b>קלט:</b> שש מסירות של חמש ביקורות ייחודיות. ארבע ביקורות שליליות שייכות למוצר 5 באותו חלון. אחת מהביקורות נמסרת פעמיים.</p>
<p><b>כלל:</b> ברגע שמספר הביקורות בדירוג 1 או 2 באותו חלון מגיע לשלוש, נשמרת התראה אחת. ביקורת רביעית וכפילות של הודעה אינן יוצרות התראה נוספת לאותו מוצר וחלון.</p>
<p>זוהי התראה מוקדמת במהלך חלון tumbling. היא אינה מייצגת בהכרח את 60 השניות האחרונות בכל רגע.</p>

</div>

In [ ]:
review_input = generate_review_fixture()
review_db = new_db("reviews")
with StreamEngine(review_db, alert_threshold=3) as engine:
    review_progress = []
    for item in review_input:
        result = engine.ingest(item.event, offset=item.offset)
        snap = engine.snapshot()
        review_progress.append({
            "offset": item.offset, "החלטה": result["status"],
            "ביקורות ייחודיות": snap["totals"]["reviews"],
            "ביקורות שליליות": snap["totals"]["negative_reviews"],
            "התראות": len(snap["alerts"]),
        })
    reviews = engine.finalize_demo()
show_table(review_progress)
show_table(reviews["alerts"], [
    ("product_id","מוצר"), ("window_start","חלון"),
    ("negative_reviews","שליליות ברגע ההתראה"), ("trigger_event_id","אירוע שהפעיל את ההתראה"),
], caption="ההתראה שנשמרה")
check("הכפילות אינה יוצרת ביקורת נוספת", reviews["totals"]["reviews"] == 5)
check("נוצרה התראה אחת בלבד", len(reviews["alerts"]) == 1)
check("ההתראה נוצרה כשמספר השליליות הגיע לשלוש", reviews["alerts"][0]["negative_reviews"] == 3)


<div dir="rtl">
<h2>שלב 14: חיפוש בנתונים שנקלטו</h2>

<p>נחפש את המילה סוללה בביקורות שכבר נשמרו. זו בדיקת חיפוש מילולי פשוטה, שמראה שהנתון החדש זמין לצרכן נוסף.</p>
<p>חיפוש זה אינו משתמש ב-embeddings ואינו חיפוש סמנטי. הרחבת RAG תוסיף יצירת וקטורים, כתיבה אידמפוטנטית לאינדקס ושמירה של המקור ושל גרסת המודל.</p>

</div>

In [ ]:
with StreamEngine(review_db, alert_threshold=3) as engine:
    matches = engine.search_reviews("סוללה")
show_table(matches, [
    ("event_id","מזהה ביקורת"), ("product_id","מוצר"),
    ("rating","דירוג"), ("text","תוכן"),
])
check("הביקורת על הסוללה זמינה לחיפוש", len(matches) == 1 and "סוללה" in matches[0]["text"])


<div dir="rtl">
<h2>שלב 15: תרגיל עצמאי</h2>

<p>לפני ההרצה נבאו מה יקרה כאשר ה-grace משתנה ל-30 שניות. החליפו גם את סף ההתראה ב-4 וב-5, והסבירו מדוע מספר ההתראות משתנה.</p>
<p><b>התוצר:</b> טבלת תוצאות והסבר קצר שכולל קלט, כלל, תוצר ובדיקה. קובץ <code dir="ltr">docs/EXERCISE_HE.md</code> מפרט את התרגיל ואת שאלות הדיון.</p>
<p>הבדיקות בתא הבא בודקות שהסכומים נשמרו ושאין התראה כפולה. הן אינן מחליפות הסבר של ההחלטה העסקית.</p>

</div>

In [ ]:
EXERCISE_GRACE = 30
EXERCISE_ALERT_THRESHOLD = 4
with StreamEngine(new_db("exercise-orders"), grace_seconds=EXERCISE_GRACE) as engine:
    for item in orders_input:
        engine.ingest(item.event, offset=item.offset)
    exercise_orders = engine.finalize_demo()
with StreamEngine(new_db("exercise-reviews"), alert_threshold=EXERCISE_ALERT_THRESHOLD) as engine:
    for item in review_input:
        engine.ingest(item.event, offset=item.offset)
    exercise_reviews = engine.finalize_demo()
show_table([{
    "grace": EXERCISE_GRACE, "הזמנות שהתקבלו": exercise_orders["totals"]["orders"],
    "הזמנות מאוחרות": exercise_orders["totals"]["late_orders"],
    "סף התראה": EXERCISE_ALERT_THRESHOLD, "התראות": len(exercise_reviews["alerts"]),
}])
check("גם בתרגיל כל הסכום נשמר",
      exercise_orders["totals"]["revenue_cents"] + exercise_orders["totals"]["late_order_cents"] == orders_truth["revenue_cents"])
check("בתרגיל אין יותר מהתראה אחת למוצר ולחלון", len(exercise_reviews["alerts"]) <= 1)


<div dir="rtl">
<h2>שלב 16: קובצי התוצאות</h2>

<p>נשמור את תוצאות ההרצה הנוכחית כטבלאות CSV, קובץ JSON ודוח HTML בעברית. קובצי SQLite נמצאים בתיקיות המשנה של אותה הרצה.</p>
<p><b>איך רואים?</b> אפשר ללחוץ על קישורי הקבצים במחברת. ב-Cursor ניתן לפתוח את תיקיית lab_runs בסייר הקבצים. את report.html פותחים בדפדפן ואת קובצי CSV אפשר לפתוח גם ב-Excel.</p>

</div>

In [ ]:
exported = save_report(
    RUN_DIR, comparison=comparison, trace=trace, teaching=teaching,
    orders=orders, reviews=reviews, live=live, checks=checks,
    microbatch_comparison=microbatch_comparison, microbatch_trace=microbatch_trace,
)
show_table(checks, caption=f"{len(checks)} בדיקות עברו בהרצה זו")
print("תיקיית התוצאות:", RUN_DIR)
for key in ("report", "results", "revenue_windows", "late_events", "review_alerts"):
    path = exported[key]
    assert path.exists() and path.stat().st_size > 0
    display(FileLink(str(path.relative_to(ROOT)), result_html_prefix=key + ": "))


<div dir="rtl">
<h2>שלב רשות: ברוקר אמיתי עם Kafka API</h2>

<p>המסלול המקומי כבר הושלם. במסלול הבא Redpanda שומר את ההודעות בלוג, והיצרן והצרכן יכולים לפעול בתהליכים נפרדים. נדרשת התקנה פועלת של Docker Desktop ב-Windows.</p>
<p>פותחים טרמינל בתיקיית הפרויקט ומריצים לפי הסדר:</p>
<pre dir="ltr">uv sync --extra kafka
docker compose up -d --wait
uv run --extra kafka python -m streaming_lab.kafka_demo topics
uv run --extra kafka python scripts/kafka_smoke.py --bootstrap localhost:19092</pre>
<p>המדריך <code dir="ltr">docs/KAFKA_LAB.md</code> מסביר גם עבודה בכמה טרמינלים, קבוצות צרכנים, commit וקריסה. לכל צרכן במעבדה יש topic אחד, partition אחד ומסד מקומי משלו.</p>
<p>התא הבא אינו מפעיל Docker אוטומטית. אחרי שהברוקר פעיל אפשר לשנות את הדגל ל-True. הוא מפעיל בדיקה אמיתית עם זמן המתנה מוגבל.</p>

</div>

In [ ]:
RUN_KAFKA_SMOKE = False
if RUN_KAFKA_SMOKE:
    import subprocess
    completed = subprocess.run(
        [sys.executable, str(ROOT / "scripts/kafka_smoke.py"), "--bootstrap", "localhost:19092"],
        cwd=ROOT, check=True, capture_output=True, text=True,
        encoding="utf-8", errors="replace", timeout=300,
    )
    print(completed.stdout)
else:
    print("מסלול הברוקר לא הופעל. המסלול המקומי והבדיקות שלו הושלמו.")


<div dir="rtl">
<h2>מקורות והמשך</h2>

<p>מימוש המעבדה: <code dir="ltr">src/streaming_lab</code>. בדיקות: <code dir="ltr">tests</code>. כללי ההרצה וההגבלות מתוארים במדריך המרצה.</p>
<ul>
<li><a href="https://docs.python.org/3/library/asyncio-queue.html">תור asyncio והמתנה כשהתור מלא</a></li>
<li><a href="https://kafka.apache.org/documentation/">מושגי Kafka וצרכנים</a></li>
<li><a href="https://docs.confluent.io/kafka-clients/python/current/overview.html">Producer ו-Consumer ב-Python</a></li>
<li><a href="https://docs.astral.sh/uv/guides/integration/jupyter/">מחברות עם uv</a></li>
<li><a href="https://www.sqlite.org/atomiccommit.html">עסקאות אטומיות ב-SQLite</a></li>
</ul>
<p>שאלת סיום: מה יקרה אם נשלח אימייל לפני ה-commit המקומי ואז התהליך יקרוס? מדוע ההגנה על הטבלאות אינה מבטיחה שהאימייל יישלח רק פעם אחת?</p>

</div>